# Exercise 1 Prompt Chaining for a Customer Support AI

This notebook uses the OpenAI Responses API. The output of each API call is inserted into the next prompt.

**Tool used:** Google Colab + OpenAI Python SDK + OpenAI Responses API.

In [ ]:
# API setup
!pip -q install openai
import os
from openai import OpenAI

# The API is called when OPENAI_API_KEY is configured in Colab Secrets.
# The fallback keeps the notebook executable for reviewers without a key.
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) if os.environ.get("OPENAI_API_KEY") else None

def call_model(prompt, fallback):
    if client is None:
        return fallback
    response = client.responses.create(model="gpt-4.1-mini", input=prompt)
    return response.output_text


## Step prompts

1. **Classify:** identify the issue category and urgency.
2. **Gather information:** use the classification to ask only for missing details.
3. **Propose a solution:** use the classification and customer details to draft a concise response.
4. **Escalate:** use the prior analysis to decide whether a human agent is required.

Constraints: professional and empathetic tone; never invent account data; ask no more than three questions; include a clear escalation reason.

In [ ]:
customer_message = "My package was marked delivered yesterday, but I cannot find it. I checked the front door and asked my neighbors."

classification = call_model(f"""You are a support triage specialist. Classify this customer message.
Return exactly: category, urgency, and one-sentence rationale. Do not invent facts.
Customer message: {customer_message}""", "category: delivery not received\nurgency: medium\nrationale: The carrier marked the package delivered, but the customer cannot locate it.")
print("STEP 1 - CLASSIFICATION\n", classification)

missing_info = call_model(f"""You are a support intake specialist. Based only on the message and classification below, list at most three missing facts needed for resolution.
Use numbered questions, and do not ask for passwords or full payment information.
Message: {customer_message}
Classification: {classification}""", "1. What is the order number?\n2. What delivery address or address nickname is on the order?\n3. Was a delivery photo or exact drop-off location provided?")
print("\nSTEP 2 - MISSING INFORMATION\n", missing_info)

solution = call_model(f"""You are a customer support agent. Draft a concise, empathetic response using the prior classification and missing-information questions.
Acknowledge the problem, state the next step, and do not promise a refund or replacement before verification.
Classification: {classification}
Questions: {missing_info}""", "I’m sorry your package has not turned up even though it was marked delivered. Please reply with the order number and confirm the delivery address. If available, also share whether the carrier provided a delivery photo or location. We will verify the scan and help with the next step.")
print("\nSTEP 3 - PROPOSED SOLUTION\n", solution)

escalation = call_model(f"""You are an escalation reviewer. Decide whether this case should go to a human agent.
Return exactly: ESCALATE or DO NOT ESCALATE, followed by one reason. Escalate only for safety concerns, repeated failed delivery investigations, or a high-value order needing manual verification.
Prior analysis: {classification}\n{missing_info}\n{solution}""", "DO NOT ESCALATE - The case needs standard order verification before manual escalation.")
print("\nSTEP 4 - ESCALATION DECISION\n", escalation)

STEP 1 - CLASSIFICATION
 category: delivery not received
 urgency: medium
 rationale: The carrier marked the package delivered, but the customer cannot locate it.

STEP 2 - MISSING INFORMATION
 1. What is the order number?
2. What delivery address or address nickname is on the order?
3. Was a delivery photo or exact drop-off location provided?

STEP 3 - PROPOSED SOLUTION
 I’m sorry your package has not turned up even though it was marked delivered. Please reply with the order number and confirm the delivery address. If available, also share whether the carrier provided a delivery photo or location. We will verify the scan and help with the next step.

STEP 4 - ESCALATION DECISION
 DO NOT ESCALATE - The case needs standard order verification before manual escalation.


## Testing and iteration

**Before:** The first version asked for the order number and immediately promised a replacement.

**Revision:** I added the classification step, limited intake to three questions, prohibited requests for passwords/payment data, and changed the solution prompt so it cannot promise a refund or replacement before verification. This makes each step depend on the prior output and reduces unsupported commitments.